[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/kerr/kerr_metric.ipynb)

# Light Past a Rotating Black Hole — Metric and Connection

Light bends around a rotating black hole, and the disk's light reaches a camera along several paths. A rotating polarizer reads the polarization carried along those paths; orbiting clumps reveal the different travel times. This notebook describes that geometry through its metric and Levi-Civita connection.

Directions are intrinsic antivectors and momentum is an extrinsic covector. The metric takes a direction to its reading, `metric(geometry)(direction)`, while its inverse takes momentum to a ray tangent. Coordinate screen directions follow `-christoffel(geometry, tangent, screen)`. Orthonormal observer frames define the camera and prescribed emission; the evolution uses the metric directly. [The GTG notebook](kerr.ipynb) describes the same scene using local screens and a bivector connection.

In tensor-index notation the Cartesian Kerr–Schild metric is $g_{\mu\nu}=\eta_{\mu\nu}-2f\ell_\mu\ell_\nu$, with $\eta=\operatorname{diag}(1,-1,-1,-1)$; its inverse is $g^{\mu\nu}=\eta^{\mu\nu}+2f l^\mu l^\nu$. The field follows [Pelle et al., equations 24–26](https://arxiv.org/pdf/2206.06429), with the overall signature reversed.


In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples,jax] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from dataclasses import dataclass

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import STA as ga
from examples.util.animation import save_figure, show_animation
from examples.relativity.kerr import render
from examples.relativity.kerr.metric import resolve, trace

In [ ]:
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
# Events and directions are antivectors; momenta, gradients and the slices that read coordinates are
# vectors, covectors that pair with directions.
Vector = ga.gatype.vector()
Bivector = ga.gatype.bivector()
Antivector = ga.gatype.antivector()
Spin = Bivector.derive.within("x y z")
Rotor = ga.gatype.rotor()
# Directions on the camera's screen, along its own x and y: the antivectors that hold both t and z.
Screen = Antivector.derive.containing("t z")
# The camera's slices x and y, which read screen directions: vectors, extrinsic.
ScreenSlice = Vector.derive.within("x y")
ScreenFrame = ga.gatype((Antivector, Screen))                # Antivector <- Screen
# The light's polarization covariance, from the slices that read the screen to screen directions.
Coherency = ga.gatype((Screen, ScreenSlice))                 # Screen <- ScreenSlice
# The metric relates complementary types; its inverse returns a direction.
Metric = ga.gatype((Vector, Antivector))                    # Vector <- Antivector
InverseMetric = ga.gatype((Antivector, Vector))             # Antivector <- Vector
# The null direction's derivative: from a step to the direction's change along it.
DirectionGradient = ga.gatype((Antivector, Antivector))      # Antivector <- Antivector
# One unit along the lab's time, the direction across the lab's slice of space `mv.t`, and the part of a
# direction within that slice.
time = -mv.t.dual()                                          # [] Antivector
space = Antivector - time * (mv.t & Antivector)              # Antivector <- Antivector
# One unit along time and along each axis of space: an event's coordinates weight them.
axes = stack([time, mv.x.dual(), mv.y.dual(), mv.z.dual()])  # [4] Antivector
# G = c = M = 1.
mass = 1.0

## 1. The Kerr–Schild metric

The scalar profile and null direction specify the metric at each event. Its null plane reads a direction, and the metric adds a null-plane contribution to the flat lowering. Nullness makes the inverse exact after one correction. A proper-time interval is `metric(geometry)(direction) & direction`; the pairing is independent of the metric that supplies its first argument.


In [ ]:
# A few events around a hole spinning in a tilted plane, at their coordinates (t, x, y, z).
event_coordinates = np.array([[0, 4, 2, 1], [3, -3, 1, 2], [-1, 2, -4, -1]])
events = (axes * event_coordinates).sum(axis=-1)                                # [events] Antivector
tilted_spin = 0.7 * mv.xy + 0.2 * mv.yz                                         # [] Spin

In [ ]:
@dataclass(frozen=True)
class Field:
    radius: Scalar
    profile: Scalar
    null: Antivector
    profile_gradient: Vector
    null_gradient: DirectionGradient

    @property
    def null_plane(self) -> Vector:
        """The null direction's plane, the light cone's tangent plane along it, which contains the
        direction: minus its dual."""
        return -self.null.dual()                                                            # [...] Vector


def field(position: Antivector, spin: Spin, mass: float) -> Field:
    """The Kerr profile, null direction and their derivatives at a batch of events."""
    spatial = space(position)                                                               # [...] Antivector
    # The volume the position spans with the spin plane: the spin times the position's height off its
    # plane, held by its complement along time. Wedged with the spin it is that height along the spin's
    # axis, times the spin squared; the spin enters twice, so its sense of turning drops out.
    volume = spatial.anticommutator(spin)                                                   # [...] Vector
    height = volume ^ spin                                                                  # [...] Antivector
    spin_squared = -spin.squared()                                                          # [...] Scalar
    offset = spatial.squared() - spin_squared                                               # [...] Scalar
    discriminant = (offset.squared() - 4 * volume.squared()).square_root()                  # [...] Scalar
    radius_squared = (offset + discriminant) / 2                                            # [...] Scalar
    radius = radius_squared.square_root()                                                   # [...] Scalar
    # Paired with a small step, how far the step changes the oblate radius.
    radius_gradient = (radius_squared * spatial - height).dual() / (radius * discriminant)  # [...] Vector
    # The hole turns as `(spin * (-angle / 2)).exp()` does, and drags the frame with it: the outgoing
    # direction twists against the turning, by the position's commutator with the spin.
    direction = (radius * spatial - spatial.commutator(spin) - height / radius) / (radius_squared + spin_squared)  # [...] Antivector
    lever = spatial - 2 * radius * direction + height / radius_squared                      # [...] Antivector
    direction_gradient = (
        lever * (radius_gradient & Antivector)
        + radius * space - space.commutator(spin)
        - (space.anticommutator(spin) ^ spin) / radius
    ) / (radius_squared + spin_squared)                                                     # [...] Antivector <- Antivector
    denominator = radius_squared.squared() - volume.squared()                               # [...] Scalar
    profile = mass * radius**3 / denominator                                                # [...] Scalar
    profile_gradient = profile * (3 * radius_gradient / radius
        - (4 * radius**3 * radius_gradient - 2 * height.dual()) / denominator)              # [...] Vector
    return Field(radius, profile, time - direction, profile_gradient, -direction_gradient)


def metric(geometry: Field) -> Metric:
    """The physical metric: a direction's covector, with positive timelike square."""
    plane = geometry.null_plane
    return -Antivector.dual() - 2 * geometry.profile * plane * (plane & Antivector)


def inverse_metric(geometry: Field) -> InverseMetric:
    """The inverse metric, taking a covector momentum to its coordinate tangent."""
    null = geometry.null
    return -Vector.dual() + 2 * geometry.profile * null * (Vector & null)


def proper_squared(geometry: Field, direction: Antivector) -> Scalar:
    """The metric's reading of a direction on itself, positive for a timelike direction."""
    return metric(geometry)(direction) & direction


def overlap_gradient(geometry: Field, momentum: Vector) -> Vector:
    """The gradient of the momentum's pairing with the null direction, `momentum & null`: the momentum
    pulled back through the null direction's derivative."""
    return geometry.null_gradient.adjugate()(momentum)                                      # [...] Vector
geometry = field(events, tilted_spin, mass)
null_squared = geometry.null.squared()
round_trip = inverse_metric(geometry)(metric(geometry)(mv.x.dual())) - mv.x.dual()


In [ ]:
with np.printoptions(precision=3, suppress=True):
    print("oblate radius:          ", geometry.radius.to_array())
    print("profile:                ", geometry.profile.to_array())
    print("null direction squared: ", null_squared.to_array())
    print("inverse metric, on x: ", round_trip.squared().to_array())

## 2. Rays from the inverse metric

A momentum reads the tangent returned by the inverse metric, and half that reading is the Hamiltonian. Differentiating with respect to momentum returns the tangent; differentiating with respect to position returns minus the momentum rate. The derivative of the null direction pulls momentum back through its adjugate, leaving the natural momentum–direction pairing visible.

In tensor-index notation, $H=\tfrac12 g^{\mu\nu}p_\mu p_\nu$, $\dot x^\mu=g^{\mu\nu}p_\nu$, and $\dot p_\mu=-\tfrac12(\partial_\mu g^{\alpha\beta})p_\alpha p_\beta$.


In [ ]:
def hamiltonian(geometry: Field, momentum: Vector) -> Scalar:
    """Half the momentum's reading of its image under the inverse metric."""
    return (momentum & inverse_metric(geometry)(momentum)) / 2


def motion(geometry: Field, momentum: Vector) -> tuple[Antivector, Vector]:
    """The two Hamiltonian derivatives at a field: the ray tangent and minus its position gradient."""
    overlap = momentum & geometry.null                                                      # [...] Scalar
    # Differentiating the quadratic Hamiltonian with respect to momentum applies the inverse metric.
    tangent = inverse_metric(geometry)(momentum)                                    # [...] Antivector
    # The gradient of profile * overlap squared, by the product rule.
    gradient = (geometry.profile_gradient * overlap.squared()
                + 2 * geometry.profile * overlap * overlap_gradient(geometry, momentum))      # [...] Vector
    return tangent, -gradient                                                               # [...] Antivector, [...] Vector


def rates(position: Antivector, momentum: Vector, spin: Spin, mass: float) -> tuple[Antivector, Vector]:
    """The two Hamiltonian derivatives: ray tangent and minus its position gradient."""
    return motion(field(position, spin, mass), momentum)                                    # [...] Antivector, [...] Vector


## 3. The same light at every spin

The same rays come in along x past a black hole whose spin sweeps from $-0.8M$ to $+0.8M$ and back, fixed along each ray. Spin changes which rays escape and how far they turn. Inside the horizon the integrator shortens its affine step and stops at a surface within; the black disk is the horizon's equatorial section, and the colours follow each ray's initial offset.

In [ ]:
n_frames = 80
max_spin = 0.8 * mass
# One sweep of the spin from one sense to the other; the animation plays it forward and back.
spins = -max_spin * np.cos(np.linspace(0, np.pi, n_frames // 2 + 1))
duration_ms = 80
extent = 12.0 * mass
n_rays = 31
offsets = np.linspace(-8, 8, n_rays) * mass
step_size, n_steps = 0.1 * mass, 400
stop_fraction = 0.8
n_ring_samples = 128

In [ ]:
spin = mv.xy * spins[:, None]                                    # [frames, 1] Spin
position = -mv.x.dual() * extent + mv.y.dual() * offsets          # [rays] Antivector
# The null correction to a flat null momentum solves the curved null condition exactly.
geometry = field(position, spin, mass)                             # [frames, rays] Field
local_null = time + mv.x.dual()                                    # [] Antivector
momentum = -local_null.dual() - geometry.profile * geometry.null_plane * (geometry.null_plane & local_null)             # [frames, rays] Vector
momentum = momentum / (momentum & time)                            # [frames, rays] Vector
position = position.broadcast_to(momentum.shape)                   # [frames, rays] Antivector
paths, _ = trace(position, momentum, spin, mass, step_size, n_steps, stop_fraction, rates)   # [steps + 1, frames, rays] Antivector

# The horizon's oblate radius gives this equatorial circle in the drawing coordinates.
horizon = mass + np.sqrt(mass**2 - spins**2)                       # [frames]
radius = np.sqrt(horizon**2 + spins**2)                            # [frames]
angles = np.linspace(0, 2 * np.pi, n_ring_samples + 1)
turns = (mv.xy * (-angles / 2)).exp()                              # [ring_samples + 1] Rotor
horizons = (turns >> mv.x.dual()) * radius[:, None]                # [frames, ring_samples + 1] Antivector
show_animation(render.animate_rays(paths, horizons, extent), "kerr_metric_spin", duration_ms, forward_then_back=True)

## 4. The metric connection carries the screen

The screen is a pair of coordinate directions. Parallel transport keeps its metric lengths and its orthogonality to the ray. The connection is constructed directly from changes of the metric: two directional derivatives, less the gradient of their scalar reading, followed by the inverse metric. Both inputs are constant coordinate directions while taking those derivatives. No local rotation field enters the evolution.

In tensor-index notation, $\Gamma^\mu{}_{\alpha\beta}=\tfrac12g^{\mu\nu}(\partial_\alpha g_{\nu\beta}+\partial_\beta g_{\nu\alpha}-\partial_\nu g_{\alpha\beta})$ and $\dot s^\mu=-\Gamma^\mu{}_{\alpha\beta}\dot x^\alpha s^\beta$.


In [ ]:
def metric_change(geometry: Field, along: Antivector, direction: Antivector) -> Vector:
    """The metric's directional derivative along a coordinate step, applied to a fixed direction."""
    plane = geometry.null_plane
    plane_change = -geometry.null_gradient(along).dual()
    return -2 * (
        (geometry.profile_gradient & along) * plane * (plane & direction)
        + geometry.profile * (plane_change * (plane & direction)
                              + plane * (plane_change & direction)))


def christoffel(geometry: Field, along: Antivector, direction: Antivector) -> Antivector:
    """The torsion-free metric connection in the fixed Cartesian coordinate basis."""
    plane = geometry.null_plane
    along_reading, direction_reading = plane & along, plane & direction
    along_gradient = geometry.null_gradient.adjugate()(-along.dual())
    direction_gradient = geometry.null_gradient.adjugate()(-direction.dual())
    # Differentiate the scalar metric reading with both coordinate directions held fixed.
    reading_gradient = -2 * (
        geometry.profile_gradient * along_reading * direction_reading
        + geometry.profile * (along_gradient * direction_reading
                              + along_reading * direction_gradient))
    # Two directional derivatives and the scalar reading's gradient give the lowered connection;
    # the inverse metric returns its direction.
    return inverse_metric(geometry)(
        metric_change(geometry, along, direction)
        + metric_change(geometry, direction, along) - reading_gradient) / 2


def polarized_rates(position: Antivector, momentum: Vector, screen: ScreenFrame,
                    spin: Spin, mass: float) -> tuple[Antivector, Vector, ScreenFrame]:
    """Hamiltonian motion and coordinate-screen transport, per unit of decreasing coordinate time."""
    geometry = field(position, spin, mass)
    tangent, momentum_rate = motion(geometry, momentum)
    screen_rate = -christoffel(geometry, tangent, screen)
    clock = -(mv.t & tangent)
    return tangent / clock, momentum_rate / clock, screen_rate / clock


## 5. The glowing disk

The gas around the hole forms a thin disk in its equatorial plane, orbiting with the hole and ending at the innermost stable circular orbit, and glows as a blackbody at the temperature of a relativistic thin disk, from the flux of [Page and Thorne (1974)](https://doi.org/10.1086/152990): zero at the innermost stable orbit, hottest a little outside it, and falling as the radius to the power −3/4 further out. Each pixel's ray is followed back until it is captured or escapes to the sky, the disk absorbing a share of it wherever the ray crosses it.

Each crossing's frequency ratio is two pairings with the ray's momentum, with the camera at rest and with the orbiting gas, and a blackbody seen at a shifted frequency is a blackbody at the shifted temperature. The camera sees the disk lensed around the shadow, its far side bent up over the top; the side coming towards the camera is bluer and brighter, the side going away redder and dimmer. The colours are exaggerated in saturation to show the shift. Tracing the camera's 480 × 320 pixels takes about a minute, almost all of it following the rays and carrying their screens.

The disk model uses prograde, subextremal spin, $0 \leq a/M < 1$. A coordinate-static camera uses the same observer for its angles, screen and frequency measurement; `camera_boost` relates its rest frame to the Kerr–Schild frame.

The orthonormal `source_frame` fixes the same angular and emission convention as the GTG notebook. A camera or source frame is also a standard part of a metric description; here it is used only to construct initial data and read emitted light.


In [ ]:
def orbiting_gas(position: Antivector, orbit: Bivector, spin: Spin, mass: float) -> Antivector:
    """The unit velocity of gas on a circular equatorial orbit in the unit plane `orbit`, at each event."""
    geometry = field(position, spin, mass)                                                  # [...] Field
    angular_speed = orbital_speed(geometry.radius, orbit, spin, mass)                       # [...] Scalar
    # A point turned by `(orbit * (-angle / 2)).exp()` moves by its commutator with the orbit's plane.
    flow = time + angular_speed * space(position).commutator(orbit)                         # [...] Antivector
    return flow / proper_squared(geometry, flow).square_root()                              # [...] Antivector

def orbital_speed(radius: Scalar, orbit: Bivector, spin: Spin, mass: float) -> Scalar:
    """Kepler's angular speed, per coordinate time, of a circular equatorial orbit in the unit plane
    `orbit` at each oblate radius, the spin counted along the orbit."""
    alignment = -spin.scalar_product(orbit)                                                 # [] Scalar
    return np.sqrt(mass) / (radius * radius.square_root() + alignment * np.sqrt(mass))      # [...] Scalar

def frequency_ratio(position: Antivector, momentum: Vector, camera: Antivector, orbit: Bivector,
                    spin: Spin, mass: float) -> Scalar:
    """The frequency a camera at rest at `camera` sees, over the one emitted by gas on a circular
    orbit in the unit plane `orbit`, for rays whose momenta are given at the gas."""
    gas = orbiting_gas(position, orbit, spin, mass)                                         # [...] Antivector
    at_rest = time / proper_squared(field(camera, spin, mass), time).square_root()         # [] Antivector
    # The momentum paired with time is conserved along the ray, so the camera's frequency reads here.
    return (momentum & at_rest) / (momentum & gas)                                          # [...] Scalar

def cube_root(value: Scalar) -> Scalar:
    """The real cube root of each non-negative scalar."""
    return (value.log() / 3).exp()

def innermost_stable_orbit(orbit: Bivector, spin: Spin, mass: float) -> Scalar:
    """The radius of the innermost stable circular equatorial orbit in the unit plane `orbit`, turning
    with the hole, the spin counted along the orbit: 6M without spin, M at the extreme (Bardeen, Press
    and Teukolsky, 1972)."""
    ratio = -spin.scalar_product(orbit) / mass                                              # [] Scalar
    first = 1 + cube_root(1 - ratio**2) * (cube_root(1 + ratio) + cube_root(1 - ratio))     # [] Scalar
    second = (3 * ratio**2 + first**2).square_root()                                        # [] Scalar
    return mass * (3 + second - ((3 - first) * (3 + first + 2 * second)).square_root())    # [] Scalar

def disk_temperature(radius: Scalar, orbit: Bivector, spin: Spin, mass: float, scale: float) -> Scalar:
    """A relativistic thin disk's temperature at each oblate radius, from the flux of Page and Thorne
    (1974), for gas orbiting in the unit plane `orbit`, the spin counted along the orbit: zero at the
    innermost stable orbit, falling as the radius to the power -3/4 far out. `scale` is
    (accretion rate c^6 / (σ G^2 M^2))^(1/4), K."""
    ratio = -spin.scalar_product(orbit) / mass                                              # [] Scalar
    root_radius = (radius / mass).square_root()                                             # [...] Scalar
    edge = (innermost_stable_orbit(orbit, spin, mass) / mass).square_root()                 # [] Scalar
    # The roots of the cubic `root_radius**3 - 3 * root_radius + 2 * ratio`, and the weight of each in
    # the torque; the product of a root's differences from the other two is the cubic's derivative there.
    roots = 2 * ((ratio.arccos() + np.array([-np.pi, np.pi, 3 * np.pi])) / 3).cos()       # [3] Scalar
    weights = (roots - ratio) ** 2 / (roots * (roots ** 2 - 1))                             # [3] Scalar
    # The torque the disk carries, zero at the innermost stable orbit; round-off kept non-negative.
    torque = (root_radius - edge - 1.5 * ratio * (root_radius / edge).log()
              - (weights * ((root_radius[..., None] - roots) / (edge - roots)).log()).sum(axis=-1)).clip(0, np.inf)  # [...] Scalar
    flux = 3 / (8 * np.pi) * torque / (root_radius ** 4 * (root_radius ** 3 - 3 * root_radius + 2 * ratio))  # [...] Scalar
    return scale * flux.square_root().square_root()                                         # [...] Scalar

def disk_depth(radius: Scalar, inner_radius: Scalar, outer_radius: float, depth: float) -> Scalar:
    """The disk's optical depth straight through its thickness at each oblate radius: `depth` at the
    inner edge, where the gas is densest, thinning outwards to none at the outer edge."""
    fraction = (radius - inner_radius) / (outer_radius - inner_radius)                      # [...] Scalar
    return depth * (1 - fraction) * (1 - fraction)                                          # [...] Scalar

def disk_opacity(position: Antivector, momentum: Vector, depth: Scalar, normal: Vector,
                 orbit: Bivector, spin: Spin, mass: float) -> Scalar:
    """The share of a ray's light the disk absorbs where the ray crosses it. Its optical depth along
    the ray is the depth through its thickness over the cosine of the ray's slant as the orbiting gas
    sees it: the ray's energy in the gas's frame over its momentum along the normal. Gas moving away
    from the light's source sees the ray more obliquely, and absorbs more."""
    gas = orbiting_gas(position, orbit, spin, mass)                                         # [...] Antivector
    slant = ((momentum & gas) / (momentum & normal.dual())).abs()                           # [...] Scalar
    return 1 - (-depth * slant).exp()                                                       # [...] Scalar


In [ ]:
camera_spin = 0.8 * mass
camera_distance = 28.0 * mass
inclination = np.deg2rad(75.0)
half_view = np.tan(np.deg2rad(19.0))
# The camera's pixels across, every ray traced.
image_width = 480
image_height = image_width * 2 // 3
step_fraction, n_camera_steps = 0.2, 720
outer_radius = 12.0 * mass
# The disk's optical depth straight through its thickness, at its inner edge.
inner_depth = 4.0
temperature_scale = 58500.0                                                         # K, (accretion rate c^6 / (σ G^2 M^2))^(1/4)
exposure = 6500.0                                                                   # K, shown at mid-grey
escape_radius = 100.0 * mass
# Captured inside every photon orbit, where no ray turns back out.
capture_radius = 1.05 * (mass + np.sqrt(mass**2 - camera_spin**2))
sky_width, sky_seed = 2048, 8
sky_height = sky_width // 2

In [ ]:
def source_frame(geometry: Field) -> DirectionGradient:
    """An orthonormal frame specifying the camera and prescribed emission, in coordinates.

    This choice fixes the same local source directions as the Kerr–Schild frame. Gravitational
    evolution uses the metric and Christoffel connection; this frame only defines local observations.
    """
    return Antivector + geometry.profile * geometry.null * (geometry.null_plane & Antivector)


def source_components(geometry: Field) -> DirectionGradient:
    """Coordinate directions resolved in the specified orthonormal source frame."""
    return Antivector - geometry.profile * geometry.null * (geometry.null_plane & Antivector)


# The camera's own axes in its rest frame, before it is turned: it looks along -z, with its right +y and its up -x.
camera_forward = -mv.z.dual()                                                    # [] Antivector
camera_right = mv.y.dual()                                                       # [] Antivector
camera_up = -mv.x.dual()                                                         # [] Antivector


def camera_boost(geometry: Field) -> Rotor:
    """A coordinate-static camera's rest frame in the Kerr–Schild frame, outside the static limit."""
    observer = source_components(geometry)(time) / proper_squared(geometry, time).square_root()  # [...] Antivector
    return (1 - observer * time).normalized()                                               # [...] Rotor


def initial_momentum(position: Antivector, direction: Antivector, spin: Spin, mass: float) -> Vector:
    """A future null covector with the given unit spatial direction in the Kerr–Schild frame."""
    geometry = field(position, spin, mass)                                                  # [...] Field
    local_null = time + direction                                                           # [...] Antivector
    # A coordinate null direction is lowered with the physical metric.
    momentum = metric(geometry)(source_frame(geometry)(local_null))                                  # [...] Vector
    return momentum / (momentum & time)                                                     # [...] Vector


def camera_rays(eye: Antivector, orientation: Rotor, pixel_width: int, pixel_height: int,
                half_view: float, spin: Spin, mass: float) -> tuple[Antivector, Vector]:
    """Pixel-centre rays of a coordinate-static camera, looking along its -z, with right +y and up -x."""
    horizontal = ((np.arange(pixel_width) + 0.5) / pixel_width * 2 - 1) * half_view * pixel_width / pixel_height
    vertical = (1 - (np.arange(pixel_height) + 0.5) / pixel_height * 2) * half_view
    direction = orientation >> (camera_forward + camera_right * horizontal[None, :] + camera_up * vertical[:, None])
    direction = direction / direction.squared().square_root()
    geometry = field(eye, spin, mass)
    # The future photon arrives opposite the direction the camera looks; its rest frame is boosted
    # into the Kerr–Schild frame before its heading is lowered into a coordinate momentum.
    heading = camera_boost(geometry) >> (time - direction)
    momentum = metric(geometry)(source_frame(geometry)(heading))
    momentum = -momentum / (momentum & time)
    momentum = momentum.reshape(-1)
    # Every ray starts at the eye, as an event of the whole of spacetime, the type its steps carry.
    return eye.broadcast_to(momentum.shape), momentum


def camera_screen(position: Antivector, momentum: Vector, orientation: Rotor,
                  spin: Spin, mass: float) -> ScreenFrame:
    """Orthonormal transverse camera axes, with horizontal x and vertical y screen inputs."""
    geometry = field(position, spin, mass)
    boost = camera_boost(geometry)
    heading = boost.reverse() >> source_components(geometry)(inverse_metric(geometry)(momentum))                                 # [...] Antivector
    direction = space(heading) / -(mv.t & heading)                                          # [...] Antivector
    # The turn carrying the camera's forward onto the direction it looks to see the ray: the reflection in
    # the forward, then in the direction halfway between the two.
    forward = orientation >> camera_forward                                                 # [] Antivector
    halfway = direction + forward                                                           # [...] Antivector
    turn = halfway * forward / halfway.squared().square_root()                              # [...] Rotor
    # The camera's own right and up, turned with it; a screen direction is read by pairing with the
    # camera's slices x and y.
    return source_frame(geometry)((boost * turn * orientation) >> (camera_right * (mv.x & Screen) + camera_up * (mv.y & Screen)))  # [...] Antivector <- Screen
orientation = (mv.zx * (-inclination / 2)).exp()                                 # [] Rotor
eye = (orientation >> mv.z.dual()) * camera_distance                             # [] Antivector
spin = mv.xy * camera_spin                                                       # [] Spin
# The thin disk ends at the innermost stable circular orbit.
inner_radius = innermost_stable_orbit(mv.xy, spin, mass)                         # [] Scalar
# Past-directed light from each pixel, and the camera's two screen axes across it.
position, momentum = camera_rays(eye, orientation, image_width, image_height, half_view, spin, mass)   # [pixels] Antivector, [pixels] Vector
screen = camera_screen(position, momentum, orientation, spin, mass)              # [pixels] Antivector <- Screen


# The share of a ray's light the disk absorbs where it crosses, at the crossing's oblate radius.
def opacity(event: Antivector, momentum: Vector, radius: Scalar) -> Scalar:
    depth = disk_depth(radius, inner_radius, outer_radius, inner_depth)
    return disk_opacity(event, momentum, depth, mv.z, mv.xy, spin, mass)


# Every ray traced to its end in one compiled loop: absorbed by the disk, escaped to the sky, or captured.
image, _ = resolve((position, momentum, screen), spin, mass, step_fraction, n_camera_steps, polarized_rates, mv.z,
                   inner_radius, outer_radius, opacity, escape_radius, capture_radius)   # Image
points, momenta, screens = image.disk                                            # [crossings] Antivector, Vector, Antivector <- Screen
# The gas orbits with the hole, in its equatorial plane.
shift = frequency_ratio(points, momenta, eye, mv.xy, spin, mass)                     # [crossings] Scalar
temperature = shift * disk_temperature(image.disk_radii, mv.xy, spin, mass, temperature_scale)   # [crossings] Scalar
sky = render.star_texture(sky_width, sky_height, sky_seed)
save_figure(render.draw_camera(image, temperature, exposure, (image_height, image_width), sky), "kerr_metric_camera");

## 6. A rotating polarizer

The disk's magnetic field circles its axis, and the light it emits is partly polarized perpendicular to that field as seen across the ray, falling smoothly to unpolarized when looking along the field. At each crossing the emission is the coherency, a covariance from the camera's slices to screen directions, built from that polarization read on the coordinate screen carried back along the ray and resolved in the source frame: the polarization weighted by each slice's reading of it, plus the unpolarized part, round, where the metric enters. An analyzer is a slice of the screen, and its reading of its own image is the power it passes, and a polarizer rotating in front of the camera picks out the pattern of the polarization across the disk. The exposure is doubled, making up for the half of unpolarized light a polarizer stops, so the starlight and the unpolarized emission look as in the still. [Gelles et al.](https://arxiv.org/abs/2105.09440) describe polarized Kerr images with prescribed emission.

In the notation of polarization optics the coherency reads as the coherency matrix $J$, and the power an analyzer along the unit vector $\mathbf a$ passes as $\mathbf a^{\mathsf T} J\,\mathbf a$.

The polarization pattern is prescribed in the local Kerr–Schild frame. It is an illustrative source model, while the orbital gas velocity determines the frequency shift and optical depth.


In [ ]:
def disk_coherency(position: Antivector, momentum: Vector, screen: ScreenFrame,
                  normal: Vector, spin: Spin, mass: float, polarization_degree: float) -> Coherency:
    """Unit-intensity disk emission resolved along the parallel-transported camera axes; `normal` is the
    disk's plane, the covector that measures height above it."""
    geometry = field(position, spin, mass)
    # The prescribed magnetic pattern is specified in this orthonormal source frame.
    heading = source_components(geometry)(inverse_metric(geometry)(momentum))
    screen = source_components(geometry)(screen)                                  # [...] Antivector
    direction = space(heading) / (mv.t & heading)                                           # [...] Antivector
    # The magnetic direction circles the normal, square to the plane through the normal and the
    # emitter. Emission is polarized perpendicular to it across the ray: in that plane, square to the ray.
    meridian = space(position).dual() ^ normal                                              # [...] Bivector
    polarization_axis = meridian.commutator(direction) / meridian.norm()                    # [...] Antivector
    polarization = screen.adjoint()(polarization_axis)                                      # [...] Screen
    # Polarization vanishes smoothly when looking along the magnetic direction.
    polarized_fraction = polarization_degree * polarization.squared()                       # [...] Scalar
    # Unpolarized light is round, where the metric enters; the polarized part weights the polarization by
    # each slice's reading of it.
    return ((1 - polarized_fraction) / 2 * ScreenSlice.dual()
            + polarization_degree * polarization * (ScreenSlice & polarization))            # [...] Screen <- ScreenSlice

def transmitted(coherency: Coherency, analyzers: ScreenSlice) -> Scalar:
    """The power each unit analyzer passes, an analyzer being the slice that reads the screen across its
    axis's normal: its reading of the coherency's image of itself."""
    return analyzers & coherency(analyzers)                                                 # [...] Scalar


In [ ]:
n_polarizer_frames = 80
polarizer_angles = np.linspace(0, np.pi, n_polarizer_frames, endpoint=False)
polarization_degree = 0.8

In [ ]:
# The emission where each ray met the disk, read through each analyzer.
coherency = disk_coherency(points, momenta, screens, mv.z, spin, mass, polarization_degree)   # [crossings] Screen <- ScreenSlice
analyzers = (mv.xy * (-polarizer_angles / 2)).exp() >> mv.x                                  # [analyzers] ScreenSlice
transmission = transmitted(coherency, analyzers[:, None])                                    # [analyzers, crossings] Scalar
show_animation(render.animate_polarizer(image, temperature, exposure, transmission, (image_height, image_width), sky),
               "kerr_metric_polarizer", duration_ms)

## 7. The disk in orbit

The disk is steady, so its light from one trace holds for every frame; only its texture moves, carried by the gas. Each frame turns every crossing's gas back along its orbit, at its own angular speed, to where it was at time zero: through the time elapsed, plus the crossing's own time, which is earlier for light that came from further away. Kepler's law makes inner gas turn several times faster than outer gas, though it moves along its orbit only about twice as fast, so the clumps wind into trailing spirals. The light delay shears the pattern further between the near and far sides. Turning the clump waves from this loop's into the previous loop's closes the loop without a jump. The clumpy texture is illustrative: it shows the flow, not a model of the gas. This cell takes about a minute, most of it drawing the 80 frames.

In [ ]:
n_flow_frames = 80
flow_time = 50.0 * mass                                                          # coordinate time the disk turns through in one loop

In [ ]:
angular_speed = orbital_speed(image.disk_radii, mv.xy, spin, mass)              # [crossings] Scalar


def disk_flow() -> Iterator[Antivector]:
    for elapsed in np.linspace(0, flow_time, n_flow_frames, endpoint=False):
        # This loop and the one before; light from further away left earlier.
        emitted = np.array([elapsed, elapsed - flow_time])[:, None] + (mv.t & points)   # [2, crossings] Scalar
        yield (mv.xy * (angular_speed * emitted / 2)).exp() >> points           # [2, crossings] Antivector


show_animation(render.animate_disk(image, temperature, exposure, disk_flow(), n_flow_frames, (image_height, image_width), sky),
               "kerr_metric_disk", duration_ms)

In [ ]:
# checks
# The null direction is null, and the metric undoes its inverse.
np.testing.assert_allclose(null_squared.kernel, 0.0, atol=1e-11)
np.testing.assert_allclose(round_trip.kernel, 0.0, atol=1e-11)
# Each pixel's light is shared at most once between the disk and the sky, all of it when its ray escapes.
n_pixels = image_width * image_height
disk_share = np.bincount(image.disk_pixels, image.disk_weights.kernel[..., 0], n_pixels)
sky_share = np.bincount(image.sky_pixels, image.sky_weights.kernel[..., 0], n_pixels)
assert np.all(disk_share + sky_share <= 1 + 1e-13)
np.testing.assert_allclose((disk_share + sky_share)[image.sky_pixels], 1, atol=1e-13, rtol=0)